# Gene Expression Classification: Does Non-Linearity Actually Help on Real Biological Data?

**Project 2 — follow-on from Project 1, testing the same question on real data.**

Project 1 showed that a linear model fails and a small neural network succeeds
on `make_circles` — clean, synthetic, low-dimensional data. This notebook asks
the same question on the dataset that motivated the field: **Golub et al.
(1999)**, gene expression profiles from 72 leukemia patients, used to
distinguish Acute Lymphoblastic Leukemia (ALL) from Acute Myeloid Leukemia
(AML).

The twist: here we have **72 samples and 3,571 features** — the opposite
regime of `make_circles`. This is where the naive "just add a neural net"
intuition gets tested against a real constraint: with more features than
samples, almost any model can perfectly fit the training data. The interesting
question is what happens on held-out patients.

**Dataset:** Golub TR, Slonim DK, Tamayo P, et al. "Molecular classification
of cancer: class discovery and class prediction by gene expression
monitoring." *Science*. 1999;286(5439):531-7.
Data source: http://hastie.su.domains/CASI_files/DATA/leukemia_small.csv
(republished by Efron & Hastie, CASI book data page).


## Step 1: Imports and setup

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, LinearSVC
from sklearn.neural_network import MLPClassifier
from sklearn.dummy import DummyClassifier

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)
RANDOM_STATE = 0


## Step 2: Load the data

Run `python data/download_data.py` first (from this project folder) to
produce `data/leukemia_clean.csv`. That script downloads the raw Golub
dataset and transposes it into the standard (samples x genes) orientation
with a `label` column.


In [ ]:
DATA_PATH = Path("data/leukemia_clean.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"{DATA_PATH} not found. Run `python data/download_data.py` from this "
        "project folder first to download and prepare the dataset."
    )

df = pd.read_csv(DATA_PATH)
X = df.drop(columns=["label"]).values
y = (df["label"] == "AML").astype(int).values  # AML=1, ALL=0

print(f"Loaded {X.shape[0]} samples, {X.shape[1]} genes.")
print(f"Class balance: ALL={np.sum(y==0)}, AML={np.sum(y==1)}")


## Step 3: Baseline — what does "guessing the majority class" get you?

With 72 samples and imbalanced classes (47 ALL / 25 AML), a trivial
always-predict-ALL classifier already scores ~65%. Any model we build has to
beat this by a meaningful margin to mean anything.

In [ ]:
dummy = DummyClassifier(strategy="most_frequent")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
dummy_scores = cross_val_score(dummy, X, y, cv=cv, scoring="accuracy")
print(f"Majority-class baseline: {dummy_scores.mean():.3f} +/- {dummy_scores.std():.3f}")


## Step 4: PCA — visualize the high-dimensional structure

With 3,571 genes and 72 samples, we can't plot the raw feature space. PCA
gives a 2D projection to see whether the two classes separate at all before
we commit to any model.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(6, 5))
for label, name, color in [(0, "ALL", "tab:blue"), (1, "AML", "tab:orange")]:
    mask = y == label
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], label=name, alpha=0.7, color=color)
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)")
ax.set_title("PCA projection: ALL vs. AML gene expression")
ax.legend()
fig.tight_layout()
fig.savefig(RESULTS_DIR / "pca_projection.png", dpi=150)
plt.show()


## Step 5: Feature selection

3,571 genes for 72 samples is a severe case of p >> n. Without reducing
dimensionality first, any classifier can trivially memorize the training set.
We use univariate F-test feature selection (ANOVA) **inside** the
cross-validation pipeline (not before it) so that no information from the
test fold leaks into feature selection — a common and easy-to-miss mistake
with high-dimensional biological data.

In [ ]:
N_FEATURES = 50  # number of top genes to keep, by ANOVA F-score

def make_pipeline(clf):
    return Pipeline([
        ("scale", StandardScaler()),
        ("select", SelectKBest(score_func=f_classif, k=N_FEATURES)),
        ("clf", clf),
    ])


## Step 6: Linear vs. non-linear models, compared with 5-fold cross-validation

72 samples is too small to trust a single train/test split (Project 1 used
one split because `make_circles` is easy; here we can't get away with that).
We use stratified 5-fold CV and report mean +/- std accuracy for each model,
all built on the *same* feature-selection pipeline so the comparison is fair.


In [ ]:
models = {
    "Logistic Regression (linear)": LogisticRegression(max_iter=5000),
    "Linear SVM": LinearSVC(max_iter=5000),
    "RBF SVM (non-linear)": SVC(kernel="rbf"),
    "MLP, 1 hidden layer of 10 (non-linear)": MLPClassifier(
        hidden_layer_sizes=(10,), max_iter=3000, random_state=RANDOM_STATE
    ),
}

cv_results = {}
for name, clf in models.items():
    pipe = make_pipeline(clf)
    scores = cross_val_score(pipe, X, y, cv=cv, scoring="accuracy")
    cv_results[name] = {"mean": scores.mean(), "std": scores.std(), "folds": scores.tolist()}
    print(f"{name:45s}  {scores.mean():.3f} +/- {scores.std():.3f}   (folds: {np.round(scores,3)})")


## Step 7: Plot the comparison

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
names = list(cv_results.keys())
means = [cv_results[n]["mean"] for n in names]
stds = [cv_results[n]["std"] for n in names]

ax.barh(names, means, xerr=stds, color="tab:blue", alpha=0.8)
ax.axvline(dummy_scores.mean(), color="red", linestyle="--",
           label=f"Majority-class baseline ({dummy_scores.mean():.2f})")
ax.set_xlabel("5-fold CV accuracy")
ax.set_xlim(0, 1.05)
ax.set_title(f"Linear vs. non-linear models on Golub leukemia data (top {N_FEATURES} genes)")
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "model_comparison.png", dpi=150)
plt.show()


## Step 8: Interpretation

Unlike `make_circles`, the honest expectation here is that **the linear
models do about as well as, or better than, the non-linear ones** — with only
72 samples and heavy feature selection, there usually isn't enough data for a
neural network's extra flexibility to pay off, and it can even overfit faster
than the linear alternatives. Whatever the actual numbers turn out to be when
you run this, that comparison — not an assumed win for neural networks — is
the real finding. Fill in the two-sentence summary below after running the
cells above:

> Fastest-to-slowest by mean CV accuracy: `<fill in after running>`.
> Margin over the majority-class baseline: `<fill in after running>`.


## Step 9: Save all results to disk

In [ ]:
summary = {
    "dataset": {
        "name": "Golub et al. 1999 leukemia gene expression (ALL vs AML)",
        "source_url": "http://hastie.su.domains/CASI_files/DATA/leukemia_small.csv",
        "n_samples": int(X.shape[0]),
        "n_genes_total": int(X.shape[1]),
        "n_genes_selected_per_fold": N_FEATURES,
        "class_counts": {"ALL": int(np.sum(y == 0)), "AML": int(np.sum(y == 1))},
    },
    "cv": {"n_splits": 5, "shuffle": True, "random_state": RANDOM_STATE},
    "majority_class_baseline_accuracy": {
        "mean": float(dummy_scores.mean()), "std": float(dummy_scores.std())
    },
    "model_comparison": cv_results,
}

with open(RESULTS_DIR / "metrics.json", "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
